# Optimize all three complex launch coefficients

Keep the same fiber and the two fixed total peak powers, **100 kW** and **81.943 kW**.
Now each launch coefficient has an amplitude and phase:

$$a_m=\frac{c_m e^{i\phi_m}}{\sqrt{\sum_jc_j^2}},\qquad P_m/P=|a_m|^2.$$

All three nonnegative amplitudes can change, with $0\le c_m\le100$ and $\sum c_m=100$.
The sum removes duplicate common scalings; it does not hold an individual amplitude fixed.
Set $\phi_1=0$ as the global phase reference and optimize $\phi_2,\phi_3$ modulo $2\pi$.
There are **six raw numbers, five stored update coordinates, and four independent
physical controls** after removing the amplitude scale and the global phase.
If a mode has zero amplitude, its phase has no physical effect.

Minimize the same all-three-mode complex-field distortion:

$$D_{\rm NL}=1-\frac{|\sum_{m=1}^{3}\int A_{\rm lin,m}^*(L,t)A_{\rm NL,m}(L,t)\,dt|^2}
{(\sum_{m=1}^{3}\int|A_{\rm lin,m}|^2dt)(\sum_{m=1}^{3}\int|A_{\rm NL,m}|^2dt)}.$$

Each candidate is sent through both the matched linear (`n2=0`) and nonlinear fiber.
This is finite-difference gradient descent of DNL, using the existing RK4IP solver.
The physical parameters and old numerical settings are unchanged. Run all cells in order
with **GNLSE (Julia)**.

In [ ]:
using Pkg
workspace = let dir = abspath(@__DIR__)
    while !isfile(joinpath(dir, "notebook_helpers", "run_library.py"))
        dirname(dir) == dir && error("Open this notebook inside the Nonlinear Optics workspace.")
        dir = dirname(dir)
    end
    dir
end
ROOT = get(ENV, "DNL_PROJECT_ROOT", joinpath(workspace, "FastDifferentiableMMNLO"))
Pkg.activate(ROOT)
using PulsePropagation, FFTW, LinearAlgebra, Printf, TOML, Dates, SHA, NPZ
using PyPlot

BLAS.set_num_threads(1)
FFTW.set_num_threads(1)
run_dir = haskey(ENV, "DNL_RUN_DIR") ? ENV["DNL_RUN_DIR"] :
    mktempdir(mkpath(joinpath(workspace, "output", "organized", "runs", "Modal amplitudes and relative phases", "Gradient descent")); prefix=Dates.format(now(), "yyyy-mm-dd") * "_step_index_phase_amplitude_optimization_", cleanup=false)
mkpath(run_dir)
println("Results: ", run_dir)

# Load the launch, objective, descent, and plotting functions used below.
include(joinpath(workspace, "notebook_helpers", "julia", "Modal amplitudes and relative phases", "shared.jl"))
include(joinpath(workspace, "notebook_helpers", "julia", "Modal amplitudes and relative phases", "step_index_phase_amplitude_optimization.jl"))

## 1. Five update coordinates, with the old setup

Use $q=(c_1/100,c_2/100,c_3/100,\phi_2,\phi_3)$, with phases measured in **radians**.
Project the first three coordinates onto the sum-one triangle and wrap the last two
into $[-\pi,\pi)$. The learning rate remains 0.10. Cap accepted amplitude changes at
10 and each phase change at **0.10 rad** (about 5.7°), using one common backtracking factor.

Amplitude finite differences still use ±0.5, checked against ±0.25. The new phase
finite differences use **±0.01 rad**, checked against ±0.005 rad. These small angular
steps are distinct from the larger allowed optimization move. A central five-component
gradient usually costs ten objective evaluations, each containing two forward solves.

Start at equal amplitudes and zero phases, as in the previous setup. A symmetric start
can have zero phase derivatives; that does not establish the best phase configuration.
This notebook performs one start at each power, not a multistart search.

Keep **200 optimization evaluations and 40 validation evaluations per power**. Charge
the starting gradient, which the optimizer reuses, to optimization; charge its half-step
check to validation. Final gradient, propagation-step, CP/dense, and time-grid checks
take priority over the old two-mode comparisons and neighbor scan. No hidden sweep is run.

In [ ]:
power_cases_W = [100_000.0, 81_943.0]
start_coefficients = fill(100.0/3, 3)
start_phases_rad = zeros(3)
phase_fd_step = 0.01                # radians; check against 0.005 rad
max_phase_change_rad = 0.10         # local angular increment, about 5.7 degrees
coefficient_bounds = (0.0, 100.0)
fd_step = 0.5                       # amplitude-coefficient units
learning_rate = 0.10                # update q = (c/100, phi2, phi3)
max_iterations = 20
max_optimizer_evaluations = 200     # per fixed-power case
max_check_evaluations = 40          # preflight + final checks, per case
max_coefficient_change = 10.0
gradient_tolerance = 1e-4           # amplitude projected gradient and phase derivatives in q
gradient_check_atol = 1e-5
gradient_check_rtol = 0.01
minimum_brightness_fraction = 0.50
search_dz = 0.005                   # 5 mm
validation_dz = 0.001               # 1 mm
dnl_tolerance = 2e-4
minimum_decrease = 1e-10

settings = Dict(
    "control"=>"three amplitudes sum(c)=100 and two relative phases; phi1=0",
    "objective"=>"all-three-mode matched linear/nonlinear complex-field DNL",
    "objective_modes"=>[1,2,3], "objective_filter"=>"none",
    "power_cases_W"=>power_cases_W, "start_coefficients"=>start_coefficients,
    "coefficient_bounds"=>collect(coefficient_bounds),
    "start_phases_rad"=>start_phases_rad, "phase_range_rad"=>[-Float64(pi),Float64(pi)],
    "phase_range_convention"=>"periodic [-pi,pi); phi1=0; absent-mode phase is immaterial",
    "phase_fd_step_rad"=>phase_fd_step,"max_phase_change_rad"=>max_phase_change_rad,
    "gradient_coordinates"=>["c1/100","c2/100","c3/100","phi2_rad","phi3_rad"],
    "finite_difference_step_coefficient"=>fd_step,
    "learning_rate_q"=>learning_rate, "max_iterations"=>max_iterations,
    "max_optimizer_evaluations_per_power"=>max_optimizer_evaluations,
    "max_check_evaluations_per_power"=>max_check_evaluations,
    "max_coefficient_change"=>max_coefficient_change,
    "gradient_tolerance_q"=>gradient_tolerance,
    "gradient_check_atol"=>gradient_check_atol, "gradient_check_rtol"=>gradient_check_rtol,
    "minimum_brightness_fraction"=>minimum_brightness_fraction,
    "brightness_definition"=>"all-mode unfiltered photons relative to equal-three-mode fine baseline at the same power",
    "search_dz_m"=>search_dz, "validation_dz_m"=>validation_dz,
    "dnl_tolerance"=>dnl_tolerance,
    "armijo_c1"=>1e-4,"max_backtracking_halvings"=>12,
    "minimum_DNL_decrease"=>minimum_decrease,
    "Nt"=>2048, "validation_Nt"=>4096, "time_window_ps"=>10.0,
    "pulse_fwhm_ps"=>0.2, "fiber_length_m"=>2.86, "carrier_m"=>1379.2e-9,
    "core_radius_m"=>52.5e-6, "NA"=>0.1, "num_modes"=>3,
    "linear_reference"=>"same candidate input and fiber; n2 = 0",
    "cp_target_error"=>1e-3, "cp_seed"=>Int(0x4d4d474e),
    "julia_version"=>string(VERSION),
    "manifest_sha256"=>bytes2hex(sha256(read(joinpath(ROOT, "Manifest.toml")))),
)
for P in power_cases_W
    mkpath(case_dir(P))
end
write_toml(joinpath(run_dir, "settings.toml"), settings)
println("Fixed powers (kW): ", power_cases_W./1000)
println("Initial coefficients: ", start_coefficients)
println("Initial phases (rad): ", start_phases_rad)

## 2. Build the same fiber and CP representation once

Mode 1 is LP01; modes 2 and 3 are the cosine and sine LP11 orientations.
Changing launch amplitudes, phases, or peak power does not change the fiber's mode profiles,
dispersion, or overlap tensor. Reuse them for both power cases.

The linear and nonlinear branches differ only in `n2`. Both score and brightness now
include all three modes. The old modes-1+2 DNL is retained only as a diagnostic.

In [ ]:
properties = compute_fiber_properties(
    StepIndex(settings["carrier_m"], settings["core_radius_m"], settings["NA"]),
    Silica(), settings["num_modes"];
    polarization=:scalar, mode_basis=:LP, beta_order=5)
S_cp = cp_compress(properties.S; error=settings["cp_target_error"], seed=settings["cp_seed"])
context_cache = Dict{Tuple{Int,Symbol},Any}()
main_context = simulation_context(settings["Nt"])
time_grid, domain = main_context.grid, main_context.domain
mode_names = ("LP01", "LP11-cos", "LP11-sin")
@printf("CP rank: %d; relative tensor error: %.4g\n", cp_rank(S_cp), S_cp.relative_error)
println("Mode labels: ", properties.modes.labels)

## 3. Evaluate a complex launch and cache the full physical input

`dnl_at(c, P; phases_rad=phi)` constructs $c_m e^{i\phi_m}$ before the pulse constructor
normalizes the amplitudes. Both propagation branches receive this same complex input.
Changing phase never changes the total peak power or the modal power fractions.

Cache keys include canonical amplitudes and relative phases, fixed power, propagation
step, time resolution, and CP/dense representation. Phases differing by $2\pi$ have
the same key. Zero-amplitude modes have no meaningful phase. The complex modal
overlaps are summed before squaring, so the score retains relative phase information.

In [ ]:
evaluation_cache = Dict{Any,NamedTuple}()
field_cache = Dict{Int,NamedTuple}()
evaluations, failures = NamedTuple[], NamedTuple[]
attempt_id = Ref(0)
used = Dict(P=>Dict(:optimization=>0, :validation=>0) for P in power_cases_W)

# Cheap checks of the constraints and the ALL-MODE overlap definition.
let
    for v in ([0.2,0.3,0.5],[-2.0,3.0,0.4],[1.0,1.0,1.0])
        u=project_simplex(v)
        @assert all(u .>= 0) && isapprox(sum(u),1.0;atol=1e-14)
        @assert isapprox(project_simplex(u),u;atol=1e-14)
    end
    @assert canonical_coefficients([20,30,50]) ≈ canonical_coefficients([40,60,100])
    a=zeros(ComplexF64,4,3,1); a[1,1,1]=1; a[1,2,1]=1
    b=copy(a); b[1,3,1]=2
    @assert field_fidelity(a,b)<1
    @assert isapprox(field_fidelity(a,2exp(0.7im).*a),1;atol=1e-12)
    for m in 1:3
        pure=zero(a); pure[1,m,1]=1
        @assert field_fidelity(pure,pure)==1
    end
end

let c=[20.0,30.0,50.0],p=[0.0,0.7,-0.4]
    @assert canonical_phases(p.+1.1)≈p
    @assert evaluation_key(c,100000.0,search_dz,2048,:cp;phases_rad=p)==
        evaluation_key(c,100000.0,search_dz,2048,:cp;phases_rad=p.+[2pi,4pi,-2pi])
    @assert evaluation_key(c,100000.0,search_dz,2048,:cp;phases_rad=[0.0,pi,0.0])==
        evaluation_key(c,100000.0,search_dz,2048,:cp;phases_rad=[0.0,-pi,0.0])
    @assert isapprox(wrap_phase(-pi+0.02-(pi-0.01)),0.03;atol=1e-12)
    empty_phase_a=gaussian_pulse(domain,3;peak_power=100000.0,fwhm=settings["pulse_fwhm_ps"],coefficients=[50.0,50.0,0.0].*cis.([0.0,0.7,1.8]))
    empty_phase_b=gaussian_pulse(domain,3;peak_power=100000.0,fwhm=settings["pulse_fwhm_ps"],coefficients=[50.0,50.0,0.0].*cis.([0.0,0.7,-0.4]))
    @assert empty_phase_a==empty_phase_b
    a=gaussian_pulse(domain,3;peak_power=100000.0,fwhm=settings["pulse_fwhm_ps"],coefficients=c.*cis.(p))
    b=gaussian_pulse(domain,3;peak_power=100000.0,fwhm=settings["pulse_fwhm_ps"],coefficients=c.*cis.(p.+0.6))
    @assert isapprox(b,cis(0.6).*a;rtol=1e-12)
    center=argmax(vec(sum(abs2.(a);dims=(2,3))))
    @assert isapprox(angle(a[center,2,1]/a[center,1,1]),p[2];atol=1e-12)
    @assert isapprox(angle(a[center,3,1]/a[center,1,1]),p[3];atol=1e-12)
    @assert norm(projected_gradient(vcat(c./100,p[2:3]),[0.0,0.0,0.0,2pi,0.0]),Inf)>6
end
println("Projection, complex-input phases, phase wrapping, cache aliases, and all-mode overlap checks passed.")

## 4. Estimate amplitude and phase derivatives

Perturb each raw amplitude by ±0.5 and each relative phase by ±0.01 rad, one at a time.
Every probe normalizes the launch, holds total power fixed, and recomputes both branches.
Multiply amplitude derivatives by 100 to obtain $dD/d(c_m/100)$; phase derivatives are
$dD/d\phi_m$ per radian. Near amplitude bounds, use second-order one-sided differences.
Phase differences remain central across the ±π boundary because the phase is periodic.

Check the tangent amplitude gradient and both phase derivatives separately, with the
same absolute-plus-relative tolerance. A large amplitude derivative must not hide an
inaccurate phase derivative. For stationarity, use the actual phase derivatives,
not the wrapped displacement of a unit gradient step.

In [ ]:
let
    for c in (0.0,0.2,50.0,99.8,100.0)
        points,weights=coefficient_stencil(c)
        @assert isapprox(sum(weights.*(points.-37).^2),2(c-37);atol=1e-9)
    end
    c=[20.0,30.0,50.0];p=[0.0,pi-0.001,-0.7];q=vcat(c./100,p[2:3])

    g=zeros(5)
    for (j,v,phi,w) in gradient_probes(q;h=0.01,hphi=1e-4)
        g[j]+=w*test_objective(v,canonical_phases(phi))
    end
    analytic=100 .* (-2c[1]^2 .* c ./ sum(abs2,c)^2)
    analytic[1]+=100*2c[1]/sum(abs2,c)
    append!(analytic,[0.2cos(p[2]),-0.3sin(p[3])])
    @assert isapprox(g,analytic;rtol=1e-6,atol=1e-8)
end
println("Amplitude and periodic phase finite-difference checks passed.")

## 5. Check the original zero-phase starts

Compare 5 mm and 1 mm DNL, then compare the full five-component gradient at
amplitude/phase steps `(0.5, 0.01 rad)` and `(0.25, 0.005 rad)`. Each power uses
its own fine-grid equal-amplitude, zero-phase baseline for the brightness floor.
The first gradient counts toward optimization and is reused in its first iteration.

In [ ]:
preflights=Dict{Float64,Any}()
brightness_floors=Dict{Float64,Float64}()
for P in power_cases_W
    baseline=dnl_at(start_coefficients,P;phases_rad=start_phases_rad)
    fine=dnl_at(start_coefficients,P;phases_rad=start_phases_rad,dz=validation_dz)
    q=controls(baseline)
    g=launch_gradient(q,P;phase=:optimization)
    gh=launch_gradient(q,P;h=fd_step/2,hphi=phase_fd_step/2)
    agreement=gradient_agreement(g,gh)
    step_error=abs(baseline.D_NL-fine.D_NL)
    ready=agreement.passed && step_error<=dnl_tolerance
    brightness_floors[P]=minimum_brightness_fraction*fine.photons
    preflights[P]=(;baseline,fine,g,gh,agreement,step_error,ready)
    write_toml(joinpath(case_dir(P),"preflight.toml"),Dict(
        "power_W"=>P,"ready"=>ready,"baseline_DNL_search"=>baseline.D_NL,
        "baseline_DNL_fine"=>fine.D_NL,"step_error"=>step_error,
        "gradient_q"=>g,"half_step_gradient_q"=>gh,
        "gradient_component_errors"=>agreement.errors,"gradient_component_thresholds"=>agreement.thresholds,
        "gradient_max_error_tolerance_ratio"=>agreement.max_ratio,
        "brightness_floor"=>brightness_floors[P]))
    @printf("%.3f kW: dz error %.3g; max gradient error/tolerance %.3g; ready=%s\n",
        P/1000,step_error,agreement.max_ratio,string(ready))
    println("  gradient [u1,u2,u3,phi2,phi3] = ",g)
end
@assert all(r.ready for r in values(preflights)) "Starting-point derivative or propagation-step checks failed."

## 6. Move amplitudes and phases together

Estimate all five derivatives, project the proposed amplitude step, and use the
negative phase gradient for the local angular step. Scale the whole direction until
the amplitude and phase caps are satisfied, then apply the existing Armijo backtracking.
The score must decrease and the all-mode brightness floor must be met.

Use the **unwrapped local angular increments** in the gradient-dot-step term. Wrap
only the stored phase coordinates. This prevents a harmless crossing of ±π from
looking like a large physical jump. Stop after 20 accepted steps, 200 optimization
evaluations, a small projected/tangent gradient, or a failed line search.

In [ ]:
optimizations=Dict(P=>optimize_launch(P) for P in power_cases_W)

## 7. Validate inside the unchanged 40-evaluation allowance

At each selected complex launch, compare 5, 2.5, and 1 mm propagation steps;
repeat the full fine-grid gradient at both perturbation sizes; compare CP with
the dense overlap tensor; and compare 2048 with 4096 time samples. Dense and
time-grid comparisons include both baseline and selected launches.

Use the last two validation evaluations for **phase diagnostic probes**: keep the
selected amplitudes fixed and shift $\phi_3$ by ±π/4 on the 1 mm grid. They can reveal
a lower score away from a symmetric phase start, but they are not optimizer steps,
stationarity proofs, or a phase sweep. They do not replace the selected descent result.

Old two-mode comparisons, a separate local amplitude-neighbor scan, LP11 rotation
checks, phase multistarts, and additional-mode convergence are marked `not_run`.
The original zero-phase amplitude notebook already showed that the old time-grid/CP
settings can miss this DNL tolerance. Keep the requested settings and report failures;
do not silently change them or promote a provisional result to a converged optimum.

In [ ]:
validations=Dict{Float64,Any}()
summaries=Dict{Float64,Any}()
comparison_rows=NamedTuple[]
for P in power_cases_W
    opt=optimizations[P];base_fine=preflights[P].fine
    c,p=coefficients(opt.best),phase_values(opt.best);q=controls(opt.best)
    best_fine=dnl_at(c,P;phases_rad=p,dz=validation_dz)
    best_mid=dnl_at(c,P;phases_rad=p,dz=search_dz/2)
    g=launch_gradient(q,P;dz=validation_dz)
    gh=launch_gradient(q,P;h=fd_step/2,hphi=phase_fd_step/2,dz=validation_dz)
    agreement=gradient_agreement(g,gh)
    pg=norm(projected_gradient(q,gh),Inf)
    dense_base=dnl_at(start_coefficients,P;phases_rad=start_phases_rad,dz=validation_dz,representation=:dense)
    dense_best=dnl_at(c,P;phases_rad=p,dz=validation_dz,representation=:dense)
    time_base=dnl_at(start_coefficients,P;phases_rad=start_phases_rad,dz=validation_dz,nt=settings["validation_Nt"])
    time_best=dnl_at(c,P;phases_rad=p,dz=validation_dz,nt=settings["validation_Nt"])
    phase_probes=NamedTuple[]
    for sign in (-1.0,1.0)
        phi=copy(p);phi[3]+=sign*pi/4
        push!(phase_probes,dnl_at(c,P;phases_rad=phi,dz=validation_dz))
    end
    save_rows(joinpath(case_dir(P),"phase_diagnostic_probes.csv"),phase_probes)
    feasible_probes=filter(r->r.photons>=brightness_floors[P],phase_probes)
    lower_phase_probe=any(r->r.D_NL<best_fine.D_NL-minimum_decrease,feasible_probes)
    phase_probe_gain=isempty(feasible_probes) ? 0.0 : max(0.0,best_fine.D_NL-minimum(r.D_NL for r in feasible_probes))
    dz_error=max(abs(opt.best.D_NL-best_fine.D_NL),abs(best_mid.D_NL-best_fine.D_NL))
    cp_error=max(abs(dense_base.D_NL-base_fine.D_NL),abs(dense_best.D_NL-best_fine.D_NL))
    time_error=max(abs(time_base.D_NL-base_fine.D_NL),abs(time_best.D_NL-best_fine.D_NL))
    history_ok=all(r->all(x->0<=x<=100,coefficients(r)) &&
        isapprox(sum(coefficients(r)),100;atol=1e-10) && phase_values(r)[1]==0 &&
        all(x->-pi<=x<pi,phase_values(r)),opt.history) &&
        all(diff([r.D_NL for r in opt.history]).<0) &&
        all(r->r.max_coefficient_step<=max_coefficient_change+1e-10 &&
            r.max_phase_step_rad<=max_phase_change_rad+1e-12,opt.history)
    checks=Dict(
        "preflight"=>preflights[P].ready,"accepted_path_constraints"=>history_ok,
        "step_size_agreement"=>dz_error<=dnl_tolerance,
        "fine_gradient_step_agreement"=>agreement.passed,
        "cp_dense_agreement"=>cp_error<=dnl_tolerance,
        "time_resolution_agreement"=>time_error<=dnl_tolerance,
        "fine_brightness"=>best_fine.photons>=brightness_floors[P])
    check_status=Dict(k=>(v ? "pass" : "fail") for (k,v) in checks)
    for omitted in ("old_two_mode_comparisons","local_amplitude_neighbors","LP11_rotation_check",
                    "phase_multistart","additional_mode_convergence")
        check_status[omitted]="not_run"
    end
    numerical_ok=all(values(checks))
    stationary=agreement.passed && pg<=gradient_tolerance
    improvement=base_fine.D_NL-best_fine.D_NL
    verified_improvement=numerical_ok && improvement>dnl_tolerance &&
        dense_best.D_NL<dense_base.D_NL && time_best.D_NL<time_base.D_NL
    summary=Dict(
        "power_W"=>P,"objective_modes"=>[1,2,3],"stop_reason"=>opt.status,
        "accepted_steps"=>length(opt.history)-1,"selected_coefficients"=>c,
        "selected_phases_rad"=>p,"selected_phases_degrees"=>rad2deg.(p),
        "selected_input_power_fractions"=>c.^2 ./ sum(abs2,c),
        "baseline_DNL_fine"=>base_fine.D_NL,"selected_DNL_fine"=>best_fine.D_NL,
        "absolute_DNL_improvement"=>improvement,"fractional_DNL_reduction"=>improvement/base_fine.D_NL,
        "brightness_ratio"=>best_fine.photons/base_fine.photons,
        "max_step_size_DNL_difference"=>dz_error,"max_cp_dense_DNL_difference"=>cp_error,
        "max_time_resolution_DNL_difference"=>time_error,
        "fine_gradient_q"=>g,"fine_half_step_gradient_q"=>gh,
        "fine_gradient_component_errors"=>agreement.errors,
        "fine_gradient_component_thresholds"=>agreement.thresholds,
        "fine_gradient_max_error_tolerance_ratio"=>agreement.max_ratio,
        "fine_projected_gradient_norm"=>pg,"projected_gradient_tolerance"=>gradient_tolerance,
        "lower_feasible_phase_probe_found"=>lower_phase_probe,"largest_phase_probe_DNL_gain"=>phase_probe_gain,
        "numerical_checks_passed"=>numerical_ok,"verified_improvement"=>verified_improvement,
        "fine_grid_stationary"=>stationary,"global_minimum_established"=>false,
        "additional_mode_convergence_checked"=>false,"application_usefulness"=>"not_specified",
        "new_optimization_evaluations"=>used[P][:optimization],"new_check_evaluations"=>used[P][:validation],
        "failed_evaluations"=>count(r->r.power_W==P,failures),"checks"=>check_status)
    summaries[P]=summary
    validations[P]=(;base_fine,best_fine,best_mid,dense_base,dense_best,time_base,time_best,
        g,gh,agreement,pg,phase_probes)
    for (label,row) in (("equal_zero_phase_baseline",base_fine),("phase_amplitude_selected",best_fine),
                       ("phase3_minus_45deg_diagnostic",phase_probes[1]),("phase3_plus_45deg_diagnostic",phase_probes[2]))
        push!(comparison_rows,comparison_row(label,row,base_fine))
    end
    write_toml(joinpath(case_dir(P),"summary.toml"),summary)
    save_rows(joinpath(case_dir(P),"comparisons.csv"),filter(r->r.power_W==P,comparison_rows))
    @printf("%.3f kW: fine DNL %.8f -> %.8f; checks=%s, stationary=%s, stop=%s\n",
        P/1000,base_fine.D_NL,best_fine.D_NL,string(numerical_ok),string(stationary),opt.status)
    println("  coefficients ",coeff_label(c),"; phases ",phase_label(p))
    @printf("  optimization %d/%d; checks %d/%d; lower phase diagnostic=%s\n",
        used[P][:optimization],max_optimizer_evaluations,used[P][:validation],max_check_evaluations,string(lower_phase_probe))
end
save_rows(joinpath(run_dir,"comparisons.csv"),comparison_rows)
write_toml(joinpath(run_dir,"summary.toml"),Dict(power_label(P)=>summaries[P] for P in power_cases_W))

## 8. Follow the amplitudes, modal powers, and phases

The first figure shows DNL, the three amplitudes, and their modal power fractions.
The phase figure shows both relative phases in degrees; the calculation uses radians.
The mixture triangle projects the saved evaluations onto modal power fractions, so
points at the same location can have different phases and different DNL. It is not
the full four-dimensional landscape or a fitted surface.

In [ ]:
mode_colors=("#366b9e","#cf7c32","#439576")
mode_styles=("-","-","--")
mode_markers=("o","s","none")
fig,axes=subplots(2,3;figsize=(16,8.4),constrained_layout=true)
for (row_index,P) in enumerate(power_cases_W)
    opt=optimizations[P];h=opt.history
    iterations=[r.iteration for r in h]
    axes[row_index,1].plot(iterations,[r.D_NL for r in h],"o-";
        color=mode_colors[1],markersize=4,label="Accepted path: 5 mm")
    axes[row_index,1].scatter([0,iterations[end]],
        [validations[P].base_fine.D_NL,validations[P].best_fine.D_NL];
        color="black",marker="x",s=60,label="Fine check: 1 mm",zorder=5)
    axes[row_index,1].set_ylabel("All-three-mode DNL")
    axes[row_index,1].set_title(@sprintf("%.3f kW | %s",P/1000,replace(opt.status,"_"=>" "));fontsize=10)
    axes[row_index,1].legend(;fontsize=8)
    for m in 1:3
        axes[row_index,2].plot(iterations,[coefficients(r)[m] for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,label=mode_names[m])
        axes[row_index,3].plot(iterations,[100coefficients(r)[m]^2/sum(abs2,coefficients(r)) for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,label=mode_names[m])
    end
    axes[row_index,2].set_ylabel("Amplitude coefficient (sum = 100)")
    axes[row_index,3].set_ylabel("Input modal power (%)")
    for j in 2:3
        axes[row_index,j].set_ylim(-2,102)
        axes[row_index,j].legend(;fontsize=8)
    end
    for j in 1:3
        axes[row_index,j].set_xlabel("Accepted iteration")
        tidy_axis(axes[row_index,j])
    end
end
fig.suptitle("Complex launch optimization: amplitude and power histories")
finish_figure(fig,joinpath(run_dir,"optimization_history"))
fig,axes=subplots(1,2;figsize=(13,5.7),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    ax=axes[j]
    rows=filter(r->r.power_W==P && r.dz==search_dz && r.nt==settings["Nt"] && r.representation=="cp",evaluations)
    xy=[triangle_xy(coefficients(r)) for r in rows]
    dots=ax.scatter(first.(xy),last.(xy);c=[r.D_NL for r in rows],cmap="viridis",s=18,alpha=0.8)
    path=[triangle_xy(coefficients(r)) for r in optimizations[P].history]
    ax.plot(first.(path),last.(path),"o-";color="#ca584b",markersize=3,label="Accepted steps",zorder=5)
    ax.scatter([first(path)[1]],[first(path)[2]];marker="o",s=85,
        facecolors="white",edgecolors="black",label="Equal-amplitude start",zorder=6)
    ax.scatter([last(path)[1]],[last(path)[2]];marker="*",s=155,
        color="#e79b36",edgecolors="black",label="Selected",zorder=7)
    ax.plot([0,1,0.5,0],[0,0,sqrt(3)/2,0];color="#888888",linewidth=1)
    ax.text(-0.03,-0.035,"LP01";ha="center",fontsize=10)
    ax.text(1.03,-0.035,"LP11-cos";ha="center",fontsize=10)
    ax.text(0.5,sqrt(3)/2+0.025,"LP11-sin";ha="center",fontsize=10)
    ax.set_aspect("equal");ax.set_xlim(-0.13,1.13);ax.set_ylim(-0.12,1.0)
    ax.axis("off")
    ax.set_title(@sprintf("%.3f kW: mixture by input power fraction",P/1000))
    ax.legend(;loc="upper left",fontsize=7.5)
    fig.colorbar(dots;ax,shrink=0.65,label="Sampled all-mode DNL (5 mm)")
end
finish_figure(fig,joinpath(run_dir,"sampled_mixture_triangles"))

fig,axes=subplots(1,2;figsize=(12.5,4.5),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    h=optimizations[P].history
    for m in 2:3
        axes[j].plot([r.iteration for r in h],[rad2deg(phase_values(r)[m]) for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,
            label=mode_names[m]*" relative to LP01")
    end
    axes[j].axhline(0;color="#aaaaaa",linewidth=0.8)
    axes[j].set_xlabel("Accepted iteration");axes[j].set_ylabel("Relative launch phase (degrees)")
    axes[j].set_title(@sprintf("%.3f kW; phi1 = 0",P/1000))
    axes[j].legend(;fontsize=8);tidy_axis(axes[j])
end
fig.suptitle("Phase coordinates are free; a symmetric start can still have small phase gradients")
finish_figure(fig,joinpath(run_dir,"phase_history"))

## 9. Compare the selected launch with the phase diagnostic probes

All bars use the 1 mm grid, 2048 time samples, and the same all-mode DNL definition.
The ±45° phase probes are diagnostic points, not additional accepted optimization steps.
The pulse plots below use the equal-amplitude zero-phase baseline and the actual
selected complex launch, with the matched linear and nonlinear outputs for each.

In [ ]:
fig,axes=subplots(1,2;figsize=(13,4.8),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    v=validations[P]
    rows=(v.base_fine,v.best_fine,v.phase_probes[1],v.phase_probes[2])
    labels=["Equal amplitudes\nzero phases","Complex launch\nselected","Diagnostic\nphi3 − 45°","Diagnostic\nphi3 + 45°"]
    bars=axes[j].bar(1:4,[r.D_NL for r in rows];color=["#a8b6c6","#366b9e","#e5bf98","#cf7c32"])
    for k in 1:4
        axes[j].text(k,rows[k].D_NL+0.012,@sprintf("%.4f",rows[k].D_NL);ha="center",fontsize=9)
    end
    axes[j].set_xticks(1:4,labels;fontsize=8)
    axes[j].set_ylim(0,min(1.05,1.17maximum(r.D_NL for r in rows)+0.02))
    axes[j].set_ylabel("All-three-mode DNL (1 mm)")
    axes[j].set_title(@sprintf("%.3f kW",P/1000));tidy_axis(axes[j])
end
finish_figure(fig,joinpath(run_dir,"phase_probe_comparisons"))

times_ps=collect(time_axis(time_grid))
wavelengths_nm=collect(wavelength_axis(time_grid,properties.omega0)).*1e9
wavelength_indices=findall(isfinite.(wavelengths_nm).&(wavelengths_nm.>0))
wavelength_indices=wavelength_indices[sortperm(wavelengths_nm[wavelength_indices])]
curve_labels=("Input","Linear reference","Nonlinear output")
curve_colors=("#777777","#cf7c32","#366b9e")
curve_styles=(":","--","-")
for P in power_cases_W
    plot_modal_output("baseline",validations[P].base_fine)
    plot_modal_output("selected",validations[P].best_fine)
end

## 10. See the spatial modes and where the energy goes

The spatial mode shapes are fixed by the fiber. Their signed field amplitudes are
shown below. The bars show how the total pulse energy is distributed among modes;
these fractions can change during nonlinear propagation.

In [ ]:
x,y=properties.modes.x,properties.modes.y
modes=properties.modes.fields # y, x, mode
let fig=figure(;figsize=(15,4.8),constrained_layout=true)
    for m in 1:3
        ax=subplot(1,3,m)
        profile=real.(modes[:,:,m]);profile ./= maximum(abs,profile)
        mesh=ax.pcolormesh(x.*1e6,y.*1e6,profile;cmap="seismic",vmin=-1,vmax=1,
            shading="auto",rasterized=true)
        theta=range(0,2pi;length=181)
        ax.plot(settings["core_radius_m"].*1e6.*cos.(theta),settings["core_radius_m"].*1e6.*sin.(theta);
            color="#555555",linestyle="--",linewidth=0.8)
        ax.set_title("Mode $m: "*mode_names[m]);ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
        fig.colorbar(mesh;ax,shrink=0.75,label="Relative field amplitude")
    end
    fig.suptitle("Fixed spatial modes; dashed circle marks the fiber core")
    finish_figure(fig,joinpath(run_dir,"spatial_modes"))
end
npzwrite(joinpath(run_dir,"spatial_mode_fields.npz"),Dict("x_m"=>x,"y_m"=>y,"modes_y_x_m"=>modes))
modal_rows=NamedTuple[]
for P in power_cases_W
    fig,axes=subplots(1,2;figsize=(12.5,4.5),constrained_layout=true,sharey=true)
    for (case_index,(label,row)) in enumerate((("Baseline",validations[P].base_fine),("Selected",validations[P].best_fine)))
        saved=field_cache[row.id]
        for (j,a) in enumerate((saved.input,saved.linear,saved.nonlinear))
            energy=vec(sum(abs2.(a);dims=(1,3)));fractions=energy./sum(energy)
            axes[case_index].bar(collect(1:3).+(j-2)*0.23,100fractions;
                width=0.23,color=curve_colors[j],label=curve_labels[j])
            for m in 1:3
                push!(modal_rows,(;power_W=P,case=lowercase(label),phase_2_rad=row.phase_2_rad,phase_3_rad=row.phase_3_rad,
                    branch=curve_labels[j],mode=m,energy_fraction=fractions[m]))
            end
        end
        axes[case_index].set_xticks(1:3,collect(mode_names))
        axes[case_index].set_ylim(0,105);axes[case_index].set_ylabel("Fraction of total pulse energy (%)")
        axes[case_index].set_title(label*" "*coeff_label(coefficients(row))*"\nphi="*phase_label(phase_values(row));fontsize=10)
        axes[case_index].legend(;fontsize=8);tidy_axis(axes[case_index])
    end
    fig.suptitle(@sprintf("%.3f kW: modal energy fractions",P/1000))
    finish_figure(fig,joinpath(case_dir(P),"modal_energy_fractions"))
end
save_rows(joinpath(run_dir,"modal_energy_fractions.csv"),modal_rows)

## 11. Reconstruct the whole beam, including interference

At the total pulse peak, add the **complex** modal fields and then square the result.
This retains interference between modes. Each branch uses its own peak time.
All beam maps below share one absolute intensity scale across both powers.
The spatial integral is checked against the summed modal peak power.

Individual-mode maps in the next cell use each mode's own peak time. They show
each mode separately and cannot be added to recover the coherent whole-beam map.

The final phase comparison holds the selected amplitudes and total input power fixed.
Only the LP11-sin launch phase changes by ±45°. These maps come from the already
computed diagnostic probes; the displayed DNL belongs to each probe's propagated
output. They show the effect of phase without confusing it with a power change.

In [ ]:
beam_maps=Dict{Tuple{Float64,String},Any}()
individual_maps=NamedTuple[]
for P in power_cases_W, (case_index,(label,row)) in enumerate((("baseline",validations[P].base_fine),("selected",validations[P].best_fine)))
    saved=field_cache[row.id]
    beam_maps[(P,label)]=[peak_beam_map(a) for a in (saved.input,saved.linear,saved.nonlinear)]
    fractions=[row.mode1_energy_fraction,row.mode2_energy_fraction,row.mode3_energy_fraction]
    for m in 1:3
        power_t=vec(sum(abs2.(saved.nonlinear[:,m,:]);dims=2));k=argmax(power_t)
        intensity=abs2.(modes[:,:,m]).*power_t[k]./1e12
        push!(individual_maps,(;power_W=P,case_index,label,mode=m,intensity,
            peak_power_W=power_t[k],time_ps=times_ps[k],energy_fraction=fractions[m]))
    end
end
observed_max=max(maximum(maximum(m.intensity) for maps in values(beam_maps) for m in maps),
    maximum(maximum(m.intensity) for m in individual_maps))
beam_color_max=max(100.0,20ceil(observed_max/20))
beam_color_ticks=collect(range(0,beam_color_max;length=6))
write_toml(joinpath(run_dir,"beam_plot_settings.toml"),Dict(
    "shared_min_W_per_um2"=>0.0,"shared_max_W_per_um2"=>beam_color_max,
    "observed_max_W_per_um2"=>observed_max))
for P in power_cases_W
    fig,axes=subplots(2,3;figsize=(15,8.6),constrained_layout=true)
    meshes=[]
    for (i,(label,row)) in enumerate((("baseline",validations[P].base_fine),("selected",validations[P].best_fine)))
        maps=beam_maps[(P,label)]
        for j in 1:3
            ax=axes[i,j];m=maps[j]
            mesh=ax.pcolormesh(x.*1e6,y.*1e6,m.intensity;cmap="magma",vmin=0,vmax=beam_color_max,
                shading="auto",rasterized=true)
            push!(meshes,mesh)
            ax.set_title(@sprintf("%s: %s\npeak %.2f kW at %.3f ps",label,curve_labels[j],m.peak_power_W/1000,m.peak_time_ps);fontsize=10)
            ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)");ax.set_aspect("equal")
        end
        npzwrite(joinpath(case_dir(P),label*"_beam_maps.npz"),Dict(
            "x_um"=>x.*1e6,"y_um"=>y.*1e6,
            "input_W_per_um2"=>maps[1].intensity,"linear_W_per_um2"=>maps[2].intensity,
            "nonlinear_W_per_um2"=>maps[3].intensity,
            "peak_times_ps"=>[m.peak_time_ps for m in maps],"peak_powers_W"=>[m.peak_power_W for m in maps],
            "shared_color_max_W_per_um2"=>beam_color_max,"launch_coefficients"=>coefficients(row),
            "launch_phases_rad"=>phase_values(row),"launch_complex_coefficients"=>coefficients(row).*cis.(phase_values(row)),
            "normalized_modal_weights"=>coefficients(row).*cis.(phase_values(row))./norm(coefficients(row))))
    end
    fig.colorbar(meshes[end];ax=vec(axes),shrink=0.8,ticks=beam_color_ticks,
        label="Intensity at each pulse's peak (W/μm²)")
    fig.suptitle(@sprintf("%.3f kW: coherent whole beam | start %s\nselected c=%s | phi=%s",
        P/1000,coeff_label(start_coefficients),coeff_label(coefficients(validations[P].best_fine)),
        phase_label(phase_values(validations[P].best_fine)));fontsize=11)
    finish_figure(fig,joinpath(case_dir(P),"baseline_selected_beams"))
end
println("Spatial integrals match summed modal peak powers for all displayed whole-beam maps.")

In [ ]:
for P in power_cases_W
    fig,axes=subplots(2,3;figsize=(15,8.6),constrained_layout=true)
    meshes=[]
    for m in filter(r->r.power_W==P,individual_maps)
        ax=axes[m.case_index,m.mode]
        mesh=ax.pcolormesh(x.*1e6,y.*1e6,m.intensity;cmap="magma",vmin=0,vmax=beam_color_max,
            shading="auto",rasterized=true)
        push!(meshes,mesh)
        ax.set_title(@sprintf("%s: %s\nenergy %.3g%% | peak %.3g kW at %.3f ps",
            m.label,mode_names[m.mode],100m.energy_fraction,m.peak_power_W/1000,m.time_ps);fontsize=10)
        ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)");ax.set_aspect("equal")
    end
    fig.colorbar(meshes[end];ax=vec(axes),shrink=0.8,ticks=beam_color_ticks,
        label="Single-mode peak intensity (W/μm²)")
    fig.suptitle(@sprintf("%.3f kW: individual nonlinear output modes",P/1000))
    finish_figure(fig,joinpath(case_dir(P),"baseline_selected_output_modes"))
end

# Reuse the already-scored phase probes; no additional objective evaluations.
phase_beam_rows=Dict(P=>(validations[P].best_fine,validations[P].phase_probes...) for P in power_cases_W)
phase_input_maps=Dict(P=>[peak_beam_map(field_cache[row.id].input) for row in phase_beam_rows[P]] for P in power_cases_W)
phase_observed_max=maximum(maximum(m.intensity) for maps in values(phase_input_maps) for m in maps)
phase_color_max=max(beam_color_max,20ceil(phase_observed_max/20))
fig,axes=subplots(2,3;figsize=(15,9.6),constrained_layout=true)
meshes=[]
phase_case_labels=("Selected phase","Diagnostic: phi3 - 45 degrees","Diagnostic: phi3 + 45 degrees")
for (i,P) in enumerate(power_cases_W)
    rows=phase_beam_rows[P];maps=phase_input_maps[P]
    for j in 1:3
        row=rows[j];p=phase_values(row);ax=axes[i,j]
        mesh=ax.pcolormesh(x.*1e6,y.*1e6,maps[j].intensity;
            cmap="magma",vmin=0,vmax=phase_color_max,shading="auto",rasterized=true)
        push!(meshes,mesh)
        ax.set_title(@sprintf("%.3f kW | %s\nphi2 %.3f deg; phi3 %.3f deg | output DNL %.5f",
            P/1000,phase_case_labels[j],rad2deg(p[2]),rad2deg(p[3]),row.D_NL);fontsize=9)
        ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)");ax.set_aspect("equal")
        if j>1
            saved=field_cache[row.id]
            label=j==2 ? "phase_probe_minus_45deg" : "phase_probe_plus_45deg"
            npzwrite(joinpath(case_dir(P),label*"_fields.npz"),Dict(
                "time_ps"=>times_ps,"wavelength_nm"=>wavelengths_nm,
                "input"=>saved.input,"linear"=>saved.linear,"nonlinear"=>saved.nonlinear,
                "power_W"=>P,"dz_m"=>row.dz,"launch_coefficients"=>coefficients(row),
                "launch_phases_rad"=>p,"launch_complex_coefficients"=>coefficients(row).*cis.(p),
                "normalized_modal_weights"=>coefficients(row).*cis.(p)./norm(coefficients(row))))
        end
    end
    npzwrite(joinpath(case_dir(P),"phase_probe_input_beams.npz"),Dict(
        "x_um"=>x.*1e6,"y_um"=>y.*1e6,"power_W"=>P,
        "intensity_W_per_um2"=>cat([m.intensity for m in maps]...;dims=3),
        "column_phase3_shift_rad"=>[0.0,-pi/4,pi/4],
        "phases_rad_by_column"=>hcat([phase_values(row) for row in rows]...),
        "launch_coefficients"=>coefficients(rows[1]),"output_DNL"=>[row.D_NL for row in rows],
        "shared_color_max_W_per_um2"=>phase_color_max))
end
fig.colorbar(meshes[end];ax=vec(axes),shrink=0.8,label="Input peak intensity (W/μm²)")
fig.suptitle("Same amplitudes and total power; change only one relative phase\nDiagnostic input beams; DNL uses the 1 mm CP propagation with Nt = 2048");
finish_figure(fig,joinpath(run_dir,"phase_probe_input_beams"))

## 12. Separate a lower score from a converged phase-and-amplitude optimum

`verified_improvement` requires the selected launch to beat its matched baseline and
pass the declared numerical checks. `fine_grid_stationary` additionally requires the
amplitude projected gradient and both phase derivatives to meet the tolerance.
The ±45° probes can reveal a lower phase point, but two probes do not settle the phase
landscape. A single zero-phase start, an iteration limit, or an evaluation limit is not
evidence of a global optimum. At zero amplitude, a mode's phase is unidentifiable.

If doubling `Nt` changes DNL too much, the old time resolution has not met the chosen
tolerance. A CP/dense failure calls for a tighter CP approximation or a dense rerun.
Disagreement between full and half perturbations calls for smaller steps before claiming
stationarity. This notebook keeps the requested parameters and marks unresolved results
as preliminary. It does not automatically enlarge budgets or relax tolerances.

CSV files store amplitudes, phases in radians, objective evaluations, accepted moves,
and rejected candidates. TOML files store controls, numerical checks, and stop reasons.
NPZ files store the actual complex fields and launch phases. The pulse and beam plots
are generated from those scored fields.

In [ ]:
for P in power_cases_W
    s=summaries[P]
    println("\n",power_label(P))
    println("  coefficients: ",coeff_label(s["selected_coefficients"]))
    println("  relative phases: ",phase_label(s["selected_phases_rad"]),"; phi1 is the reference")
    fractions=s["selected_input_power_fractions"]
    @printf("  input power fractions: [%.3f%%, %.3f%%, %.3f%%]\n",100fractions[1],100fractions[2],100fractions[3])
    @printf("  all-mode fine DNL: %.8f -> %.8f (%.2f%% reduction)\n",
        s["baseline_DNL_fine"],s["selected_DNL_fine"],100s["fractional_DNL_reduction"])
    println("  stop: ",s["stop_reason"],"; accepted steps: ",s["accepted_steps"])
    println("  numerical checks passed: ",s["numerical_checks_passed"],
        "; verified improvement: ",s["verified_improvement"],"; fine-grid stationary: ",s["fine_grid_stationary"])
    failed_checks=sort([name for (name,status) in s["checks"] if status=="fail"])
    if !isempty(failed_checks)
        println("  PRELIMINARY: failed checks = ",join(failed_checks,", "))
    end
    @printf("  max DNL differences: dz %.3g; CP/dense %.3g; Nt %.3g (tolerance %.3g)\n",
        s["max_step_size_DNL_difference"],s["max_cp_dense_DNL_difference"],
        s["max_time_resolution_DNL_difference"],dnl_tolerance)
    @printf("  projected gradient: %.6g (tolerance %.6g); lower ±45° phase probe: %s\n",
        s["fine_projected_gradient_norm"],gradient_tolerance,string(s["lower_feasible_phase_probe_found"]))
    @printf("  new evaluations: optimization %d/%d; preflight + checks %d/%d\n",
        used[P][:optimization],max_optimizer_evaluations,used[P][:validation],max_check_evaluations)
end
println("\nResults: ",run_dir)